# Summarize-then-Translate Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local Summarize-then-Translate baseline for English-to-Chinese cross-lingual dialogue summarization.

The ST pipeline uses two local small language model agents. Agent 1 reads the original English dialogue and generates a concise English summary. Agent 2 then translates the English summary into Chinese. The final output is a concise Chinese summary.

```text
English Dialogue
→ Agent 1: English Summarization Agent
→ Agent 2: Chinese Translation Agent
→ Final Chinese Summary
```

The pipeline consists of two agents:

```text
Agent 1: English Summarization Agent
Input: original English dialogue
Output: concise English summary

Agent 2: Chinese Translation Agent
Input: English summary from Agent 1
Output: final Chinese summary
```
This setup is used as a Summarize-then-Translate baseline. Unlike the Direct pipeline, ST explicitly creates an intermediate English summary before producing the Chinese summary. This allows us to inspect whether errors come from the summarization stage or the translation stage.

The local small language model is served through Ollama. The notebook controls the prompt design, agent workflow, input/output processing, intermediate output inspection, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the local model used for the Summarize-then-Translate baseline.

### Recommended model setup

This notebook uses one local small language model through Ollama for both agents:

- Agent 1: English Summarization Agent
- Agent 2: Chinese Translation Agent

```bash
ollama pull qwen3.5:27b
```

If qwen3.5:27b is too slow on your machine, you can use a smaller model for testing:

```bash
ollama pull "qwen3.5:9b"
```

Make sure the model names in the notebook match the models installed in Ollama:

```bash
SUMMARIZATION_MODEL = "qwen3.5:27b"
TRANSLATION_MODEL = "qwen3.5:27b"
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [ ]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# ST models
SUMMARIZATION_MODEL = "qwen3.5:27b"
TRANSLATION_MODEL = "qwen3.5:27b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# Gold set path
GOLD_SET_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_results/gold_set_50_zh_XSAMSum_bart.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files for ST baseline
FULL_OUTPUT_PATH = OUTPUT_DIR / "st_qwen27b_50samples.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "st_qwen27b_50samples.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "st_qwen27b_50samples_errors.jsonl"

print("Gold set path:", GOLD_SET_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Gold set path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_results/gold_set_50_zh_XSAMSum_bart.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/st_qwen27b_5samples_1.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/st_qwen27b_5samples_1.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/st_qwen27b_5samples_errors_1.jsonl


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['qwen3.5:9b', 'qwen3.5:27b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [6]:
# Cell 5: ST prompt templates

ENGLISH_SUMMARIZATION_PROMPT = """You are an English dialogue summarization agent.

Your task is to read the following English dialogue and generate a concise English summary.

Requirements:
- Summarize the main information in the dialogue.
- Write the summary in English.
- Keep the summary concise and faithful to the dialogue.
- Do not translate into Chinese.
- Do not add information that is not stated or clearly implied.
- Do not explain your reasoning.
- Output only the English summary.

Conciseness: 
- For simple dialogues, write one short English sentence.
- For complex dialogues, write at most two short English sentences.

English dialogue:
{dialogue}

English summary:
"""


CHINESE_TRANSLATION_PROMPT = """You are a Chinese translation agent.

Your task is to translate the English summary into Chinese.

Requirements:
- Translate the English summary into natural Chinese.
- Preserve the meaning of the English summary.
- Do not add new information.
- Do not remove important information.
- Do not explain your reasoning.
- Output only the final Chinese summary.
- Translate all English proper nouns, including speaker names, into standard Chinese transliteration.

STRICT TRANSLATION RULE:
- You MUST translate ALL English proper nouns and speaker names into standard Chinese characters.
- ABSOLUTELY NO English letters or names should appear in the final Chinese summary.

English summary:
{english_summary}

Chinese summary:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: ST agent functions

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace named placeholders in the prompt."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def english_summarization_agent(dialogue: str) -> str:
    """Agent 1: English dialogue -> English summary."""
    prompt = fill_prompt(
        ENGLISH_SUMMARIZATION_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=SUMMARIZATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()


def chinese_translation_agent(english_summary: str) -> str:
    """Agent 2: English summary -> Chinese summary."""
    prompt = fill_prompt(
        CHINESE_TRANSLATION_PROMPT,
        {
            "english_summary": english_summary,
        },
    )

    response = call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: ST pipeline

def run_st_pipeline(example: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
    """Run the Summarize-then-Translate pipeline."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: English dialogue -> English summary
    english_summary = english_summarization_agent(dialogue)

    if verbose:
        print("\n" + "=" * 80)
        print(f"Sample ID: {sample_id}")
        print("=== Agent 1 Output: English Summary ===")
        print(english_summary)
        print("=" * 80 + "\n")

    # Agent 2: English summary -> Chinese summary
    final_chinese_summary = chinese_translation_agent(english_summary)

    if verbose:
        print("=== Agent 2 Output: Chinese Summary ===")
        print(final_chinese_summary)
        print("=" * 80 + "\n")

    return {
        "id": sample_id,
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,

        # Intermediate output from Agent 1
        "english_summary": english_summary,

        # Final output from Agent 2
        "final_summary": final_chinese_summary,

        # References
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,

        # Metadata
        "pipeline": "summarize_then_translate",
        "summarization_model": SUMMARIZATION_MODEL,
        "translation_model": TRANSLATION_MODEL,
        "num_model_calls": 2,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect whether errors come from the English summarization stage or the Chinese translation stage.

In [ ]:
# Cell 9: Load first 5 examples from the gold set

def load_examples_from_gold_set(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the first n examples from the gold-set JSON file."""
    if not path.exists():
        raise FileNotFoundError(f"Gold set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"gold_{i+1:05d}",
            "test_index": item.get("test_index", ""),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_gold_set(GOLD_SET_PATH, n=50)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'gold_00001', 'test_index': 23, 'dialogue': "Anne: You were right, he was lying to me :/\nIrene: Oh no, what happened?\nJane: who? that Mark guy?\nAnne: yeah, he told me he's 30, today I saw his passport - he's 40\nIrene: You sure it's so important?\nAnne: he lied to me Irene", 'reference_english_summary': 'Mark lied to Anne about his age. Mark is 40.', 'reference_chinese_summary': '马克向安妮隐瞒了自己的年龄。他40岁了。'}


In [11]:
# Cell 10: Run the ST pipeline for the first example

result = run_st_pipeline(test_data[4], verbose=True)
result


Sample ID: gold_00005
=== Agent 1 Output: English Summary ===
Joyce shares a link to a cheap offer, prompting Edson to immediately book a ticket.

=== Agent 2 Output: Chinese Summary ===
乔伊斯分享了一个廉价优惠的链接，促使埃德森立即预订了机票。



{'id': 'gold_00005',
 'test_index': 66,
 'dialogue': "Joyce: Check this out!\r\nJoyce: <link>\r\nMichael: That's cheap!\r\nEdson: No way! I'm booking my ticket now!! ",
 'english_summary': 'Joyce shares a link to a cheap offer, prompting Edson to immediately book a ticket.',
 'final_summary': '乔伊斯分享了一个廉价优惠的链接，促使埃德森立即预订了机票。',
 'reference_english_summary': 'Edson is booking his ticket now.',
 'reference_chinese_summary': '埃德森正在订票。',
 'pipeline': 'summarize_then_translate',
 'summarization_model': 'qwen3.5:27b',
 'translation_model': 'qwen3.5:27b',
 'num_model_calls': 2}

In [18]:
# Cell 11: Print ST pipeline result clearly

def print_st_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1 Intermediate Output: English Summary ===")
    print(result["english_summary"])
    print()

    print("=== Agent 2 Final Output: Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Summarization model:", result["summarization_model"])
    print("Translation model:", result["translation_model"])
    print("Model calls:", result["num_model_calls"])


print_st_result(result)

=== Original Dialogue ===
Laura: Where are you?
Paul: Almost there.
Laura: Which is?
Paul: Close to the Mac.
Laura: That's so far away!
Paul: 15 mins
Laura: I am not waiting any more, see you some other time.
Paul: Please, wait!
Laura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.
Paul: I am so sorry.
Laura: I am not. 

=== Agent 1 Intermediate Output: English Summary ===
Laura cancels her meeting with Paul after waiting 30 minutes, expressing frustration that he is still 15 minutes away despite claiming to be almost there earlier. Although Paul apologizes, Laura refuses to wait any longer and decides to meet another time.

=== Agent 2 Final Output: Chinese Summary ===
劳拉在等待 30 分钟后取消了与保罗的会面，她感到沮丧，因为保罗尽管之前声称快到了，但此时距离见面仍有 15 分钟。尽管保罗道歉，劳拉拒绝再等，决定改日再约。

=== Reference English Summary ===
Paul is late for a meeting with Laura and she refuses to wait any longer.

=== Reference Chinese Summary ===
保罗和劳拉见面时迟到了，现在劳拉不想再等了。

=== Metadata ===
Pipeline: summ

## 4. Save Results

This saves the intermediate English summary and the final Chinese summary.

In [19]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/st_qwen27b_5samples.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/st_qwen27b_5samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/st_qwen27b_5samples_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed ST result to the JSONL output file.

If the notebook stops, already processed examples remain saved.

In [ ]:
# Cell 13: Batch inference with ST pipeline
# Time stamp: 1m 29.8s

MAX_EXAMPLES = 50
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running ST pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        print(f"Skipping already processed sample: {sample_id}")
        continue

    try:
        record = run_st_pipeline(ex, verbose=True)

        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "test_index": ex.get("test_index", ""),
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running ST pipeline:   0%|          | 0/5 [00:00<?, ?it/s]


Sample ID: gold_00001
=== Agent 1 Output: English Summary ===
Laura cancels her meeting with Paul after waiting 30 minutes, despite his claim 15 minutes prior that he was nearly at the Mac location. Although Paul apologizes for the delay, Laura refuses to wait any longer and ends the conversation.

=== Agent 2 Output: Chinese Summary ===
尽管保罗在15分钟前声称自己已快到达麦考尔地点，劳拉在等待30分钟后仍取消了与他的会面。虽然保罗为延误道歉，但劳拉拒绝再等，并结束了对话。


Sample ID: gold_00002
=== Agent 1 Output: English Summary ===
Finn invites Zadie to visit the Latin American community in Elephant and Castle tomorrow before it is demolished. They agree to meet at the main entrance of the shopping center at 2:00 PM to explore the area and try the local cuisine.

=== Agent 2 Output: Chinese Summary ===
Finn 邀请 Zadie 明天在 Elephant and Castle 的拉丁美洲社区被拆除前去参观。两人约定下午 2 点在购物中心主入口见面，一起探索该区域并品尝当地美食。


Sample ID: gold_00003
=== Agent 1 Output: English Summary ===
Josh asked Brian for advice on buying an iPad, but Brian advised against Apple due to its premi

## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.

For the ST pipeline, the CSV also includes the intermediate English summary from Agent 1.

In [21]:
# Cell 14: Export ST summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "test_index": record.get("test_index", ""),
        "dialogue": record.get("dialogue", ""),

        # Intermediate output from Agent 1
        "english_summary": record.get("english_summary", ""),

        # Final output from Agent 2
        "final_summary": record.get("final_summary", ""),

        # References
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),

        # Metadata
        "pipeline": record.get("pipeline", "summarize_then_translate"),
        "summarization_model": record.get("summarization_model", ""),
        "translation_model": record.get("translation_model", ""),
        "num_model_calls": record.get("num_model_calls", 2),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/st_qwen27b_5samples.csv


,id,test_index,dialogue,english_summary,final_summary,reference_english_summary,reference_chinese_summary,pipeline,summarization_model,translation_model,num_model_calls
0,gold_00001,59,Laura: Where are you?\r\nPaul: Almost there.\r...,Laura cancels her meeting with Paul after wait...,尽管保罗在15分钟前声称自己已快到达麦考尔地点，劳拉在等待30分钟后仍取消了与他的会面。虽然...,Paul is late for a meeting with Laura and she ...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。,summarize_then_translate,qwen3.5:27b,qwen3.5:27b,2
1,gold_00002,81,Finn: Hey\r\nZadie: Hi there! What's up?\r\nFi...,Finn invites Zadie to visit the Latin American...,Finn 邀请 Zadie 明天在 Elephant and Castle 的拉丁美洲社区被...,Finn and Zadie are going to Elephant and Castl...,费恩和查蒂明天2点去象堡，他们会在正门碰头。,summarize_then_translate,qwen3.5:27b,qwen3.5:27b,2
2,gold_00003,85,Josh: I need to buy an iPad?\r\nJosh: do u thi...,"Josh asked Brian for advice on buying an iPad,...",Josh 向 Brian 咨询购买 iPad 的建议，但 Brian 因苹果产品价格过高而不...,Josh wants to buy a tablet and doesn't know wh...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...,summarize_then_translate,qwen3.5:27b,qwen3.5:27b,2
3,gold_00004,87,Frank: wat are u doing??\r\nAndy: watching Arr...,Frank reminds Andy about a quiz scheduled for ...,弗兰克提醒安迪明天有测验，敦促他立即开始复习。安迪不以为意，认为测验分量不大，打算明天再复习...,Frank tries to encourage Andy to learn for the...,弗兰克试图激励安迪，为明天的测验学习。,summarize_then_translate,qwen3.5:27b,qwen3.5:27b,2
4,gold_00005,123,Crystal: <file_photo>\r\nIrene: He's so big!\r...,"Crystal shares photos of her son, noting that ...",克里斯蒂尔分享了儿子的照片，提到他的衣服已经穿不下了。艾琳主动提出带他去购物，并承诺会给他买...,Irene will take Crystal's son shopping for clo...,艾琳会带克里斯特尔的儿子去买衣服。,summarize_then_translate,qwen3.5:27b,qwen3.5:27b,2


In [22]:
# Cell 15: Compare ST outputs with references

comparison_columns = [
    "id",
    "test_index",
    "english_summary",
    "reference_english_summary",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,test_index,english_summary,reference_english_summary,final_summary,reference_chinese_summary
0,gold_00001,59,Laura cancels her meeting with Paul after wait...,Paul is late for a meeting with Laura and she ...,尽管保罗在15分钟前声称自己已快到达麦考尔地点，劳拉在等待30分钟后仍取消了与他的会面。虽然...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,81,Finn invites Zadie to visit the Latin American...,Finn and Zadie are going to Elephant and Castl...,Finn 邀请 Zadie 明天在 Elephant and Castle 的拉丁美洲社区被...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,85,"Josh asked Brian for advice on buying an iPad,...",Josh wants to buy a tablet and doesn't know wh...,Josh 向 Brian 咨询购买 iPad 的建议，但 Brian 因苹果产品价格过高而不...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,87,Frank reminds Andy about a quiz scheduled for ...,Frank tries to encourage Andy to learn for the...,弗兰克提醒安迪明天有测验，敦促他立即开始复习。安迪不以为意，认为测验分量不大，打算明天再复习...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,123,"Crystal shares photos of her son, noting that ...",Irene will take Crystal's son shopping for clo...,克里斯蒂尔分享了儿子的照片，提到他的衣服已经穿不下了。艾琳主动提出带他去购物，并承诺会给他买...,艾琳会带克里斯特尔的儿子去买衣服。


In [23]:
# Cell 16: Inspect ST outputs

if not df.empty:
    inspection_columns = [
        "id",
        "test_index",
        "dialogue",
        "english_summary",
        "reference_english_summary",
        "final_summary",
        "reference_chinese_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,test_index,dialogue,english_summary,reference_english_summary,final_summary,reference_chinese_summary
0,gold_00001,59,Laura: Where are you?\r\nPaul: Almost there.\r...,Laura cancels her meeting with Paul after wait...,Paul is late for a meeting with Laura and she ...,尽管保罗在15分钟前声称自己已快到达麦考尔地点，劳拉在等待30分钟后仍取消了与他的会面。虽然...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,81,Finn: Hey\r\nZadie: Hi there! What's up?\r\nFi...,Finn invites Zadie to visit the Latin American...,Finn and Zadie are going to Elephant and Castl...,Finn 邀请 Zadie 明天在 Elephant and Castle 的拉丁美洲社区被...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,85,Josh: I need to buy an iPad?\r\nJosh: do u thi...,"Josh asked Brian for advice on buying an iPad,...",Josh wants to buy a tablet and doesn't know wh...,Josh 向 Brian 咨询购买 iPad 的建议，但 Brian 因苹果产品价格过高而不...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,87,Frank: wat are u doing??\r\nAndy: watching Arr...,Frank reminds Andy about a quiz scheduled for ...,Frank tries to encourage Andy to learn for the...,弗兰克提醒安迪明天有测验，敦促他立即开始复习。安迪不以为意，认为测验分量不大，打算明天再复习...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,123,Crystal: <file_photo>\r\nIrene: He's so big!\r...,"Crystal shares photos of her son, noting that ...",Irene will take Crystal's son shopping for clo...,克里斯蒂尔分享了儿子的照片，提到他的衣服已经穿不下了。艾琳主动提出带他去购物，并承诺会给他买...,艾琳会带克里斯特尔的儿子去买衣服。
